# 1.0 Sobre o Notebook

- O objetivo deste notebook é fazer o reconhecimento inicial das tabelas do dataset Olist
- Fonte dos dados: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
- As tabelas encontram-se na pasta data/raw/

# 2.0 Problema de Negócio

- A Olist apresenta crescimento nas vendas e satisfação de clientes que justifiquem investimentos na expansão da plataforma?
    - As vendas estão crescendo? -> Analisar a evolução mensal dos pedidos e valor vendido
    - Os clientes estão satisfeitos? -> Analisar review score
    - O que prejudica a satisfação de um cliente? -> Analisar prazo de entrega x review_score
    - Onde estão as oportunidades? -> Olhar categorias e regiões com melhor/pior desempenho

# 3.0 Imports

In [1]:
import pandas as pd

# 4.0 Funções

In [2]:
def recon(df, nome):
    """Faz o reconhecimento inicial da tabela e exibe o resumo na tela
    
    Args:
        df (pd.DataFrame): indicar qual tabela será analisada
        nome (str): dar um nome para a análise, pode ser o mesmo nome da tabela

    Returns:
        None
    """
    print(f"========== {nome.upper()} ==========")
    print("Shape (linhas e colunas):", df.shape)
    print("Linhas duplicadas:", df.duplicated().sum()) 

    # dataframe contendo um resumo de cada coluna da tabela
    resumo = pd.DataFrame({
        "tipo": df.dtypes,
        "nulos": df.isna().sum(),
        "% nulos": (df.isna().mean() * 100).round(2),
        "únicos": df.nunique()
    })
    print("\nResumo por coluna:")
    display(resumo)

    print("Primeiras 5 linhas:")
    display(df.head())

    print("Estatística descritiva:")
    display(df.describe())

# 5.0 Carregamento dos Dados Brutos

In [3]:
df_pedidos = pd.read_csv('../data/raw/olist_orders_dataset.csv')
df_clientes = pd.read_csv('../data/raw/olist_customers_dataset.csv')
df_localizacao = pd.read_csv('../data/raw/olist_geolocation_dataset.csv') 
df_itens = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
df_pagamentos = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
df_avaliacoes = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
df_produtos = pd.read_csv('../data/raw/olist_products_dataset.csv')
df_vendedores = pd.read_csv('../data/raw/olist_sellers_dataset.csv')


# 6.0 Reconhecimento das Tabelas

## 6.1 Tabela de Pedidos

In [4]:
recon(df_pedidos, "pedidos")

========== PEDIDOS ==========
Shape (linhas e colunas): (99441, 8)
Linhas duplicadas: 0

Resumo por coluna:


,tipo,nulos,% nulos,únicos
order_id,object,0,0.00,99441
customer_id,object,0,0.00,99441
order_status,object,0,0.00,8
order_purchase_timestamp,object,0,0.00,98875
order_approved_at,object,160,0.16,90733
order_delivered_carrier_date,object,1783,1.79,81018
order_delivered_customer_date,object,2965,2.98,95664
order_estimated_delivery_date,object,0,0.00,459


Primeiras 5 linhas:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


Estatística descritiva:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,66dea50a8b16d9b4dee7af250b4be1a5,edb027a75a1449115f6b43211ae02a24,delivered,2018-08-02 12:05:26,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-08 19:36:48,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


### Investigação da Tabela de Pedidos

In [5]:
# contagem dos pedidos de acordo com status e cálculo percentual sobre o total

status_pedidos = pd.DataFrame({
    "quantidade": df_pedidos["order_status"].value_counts(),
    "percentual": (df_pedidos["order_status"].value_counts(normalize=True)*100).round(2)})

status_pedidos

,quantidade,percentual
order_status,,
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.30
created,5,0.01
approved,2,0.00


In [6]:
nao_entregue = status_pedidos.index != 'delivered' # filtra todos os pedidos diferentes de 'delivered'
print(f"Soma dos pedidos com status diferente de 'delivered': {status_pedidos.loc[nao_entregue, 'quantidade'].sum()}")


Soma dos pedidos com status diferente de 'delivered': 2963


In [7]:
# analisando nulos na coluna data de ordens aprovadas

data_nula = df_pedidos['order_approved_at'].isna()
df_pedidos.loc[data_nula, 'order_status'].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [8]:
# analisando nulos na coluna data de entrega para a transportadora

data_nula = df_pedidos['order_delivered_carrier_date'].isna()
df_pedidos.loc[data_nula, 'order_status'].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [9]:
# analisando nulos na coluna data de entrega para o cliente

data_nula = df_pedidos['order_delivered_customer_date'].isna()
df_pedidos.loc[data_nula, 'order_status'].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [10]:
# analisando o período mês a mês (há meses vazios nas pontas?)
temp_meses = pd.to_datetime(df_pedidos['order_purchase_timestamp'])

print(f'Data inicial: {temp_meses.min()} e Data final: {temp_meses.max()}')

Data inicial: 2016-09-04 21:15:19 e Data final: 2018-10-17 17:30:18


In [11]:
# quantidade de pedidos mês a mês em ordem cronológica
temp_meses.dt.to_period('M').value_counts().sort_index()

order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64

### Achados na Tabela de Pedidos:
- Existem cinco colunas de data que estão como "object" -> ajustar na etapa de limpeza dos dados
- *order_id* é único (99.441 linhas = 99.441 únicos) -> chave primária, 1 linha por pedido
- *customer_id* também apresenta 99.441 valores únicos e identifica o pedido, não a pessoa -> olhar os achados da Tabela de Clientes
- *order_purchase_timestamp* e *order_estimated_delivery_date* não tem nulos -> usadas como data de venda e prazo prometido para o cliente
- *order_purchase_timestamp* cobre de 04/09/2016 a 17/10/2018, mas o volume não é homogêneo:
    - Meses com dados residuais: 2016-09 (4 pedidos), 2016-10 (324), 2016-12 (1), 2018-09 (16) e 2018-10 (4); 2016-11 não possui registros
    - Incluir esses meses distorceria a série, sugerindo uma queda abrupta de vendas no final de 2018
    - Pico em 2017-11 (7.544 pedidos vs. 4.631 em 2017-10) -> hipótese: efeito da Black Friday
    - Decisão: período de análise de 2017-01 a 2018-08 (20 meses); crescimento medido comparando o acumulado de jan–ago/2017 com jan–ago/2018, neutralizando a sazonalidade de fim de ano
    - Contagem feita com todos os status; confirmar o recorte após o filtro 'delivered'
- *order_status* apresenta 8 possibilidades -> 'delivered' representa mais de 97% dos dados e será o único status utilizado nas análises, pois trata de pedidos com ciclo completo que permitem avaliar a experiência inteira. Observação: avaliações de pedidos cancelados ou indisponíveis ficam fora com isso, os índices de satisfação podem ser um pouco melhores do que seriam ao incluir pedidos com status diferente de 'delivered'
- *order_approved_at* possui 160 valores nulos, 14 constam como 'delivered' (inconsistente) -> coluna fora das análises, sem impacto 
- *order_delivered_carrier_date* possui 1.783 valores nulos, 2 constam como 'delivered' (inconsistente) -> coluna fora das análises, sem impacto 
- *order_delivered_customer_date* possui 2.965 valores nulos:
    - Inconsistências: 8 pedidos 'delivered' sem data de entrega; e 6 pedidos 'canceled' com data preenchida que serão excluídos pelo filtro de status (625 - 619)
    - Essa coluna será usada nas análises de prazo, os 8 casos inconsistentes (0,008% da base) serão excluídos das análises de prazo

## 6.2 Tabela de Clientes

In [12]:
recon(df_clientes, "clientes")

========== CLIENTES ==========
Shape (linhas e colunas): (99441, 5)
Linhas duplicadas: 0

Resumo por coluna:


,tipo,nulos,% nulos,únicos
customer_id,object,0,0.0,99441
customer_unique_id,object,0,0.0,96096
customer_zip_code_prefix,int64,0,0.0,14994
customer_city,object,0,0.0,4119
customer_state,object,0,0.0,27


Primeiras 5 linhas:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


Estatística descritiva:


,customer_zip_code_prefix
count,99441.000000
mean,35137.474583
std,29797.938996
min,1003.000000
25%,11347.000000
50%,24416.000000
75%,58900.000000
max,99990.000000


### Investigação da Tabela de Clientes

In [13]:
# conferindo os estados na tabela
df_clientes['customer_state'].unique()

array(['SP', 'SC', 'MG', 'PR', 'RJ', 'RS', 'PA', 'GO', 'ES', 'BA', 'MA',
       'MS', 'CE', 'DF', 'RN', 'PE', 'MT', 'AM', 'AP', 'AL', 'RO', 'PB',
       'TO', 'PI', 'AC', 'SE', 'RR'], dtype=object)

In [14]:
# quantidade de clientes por estado da federação
df_clientes['customer_state'].value_counts()

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

In [15]:
# diferença entre customer_id e customer_unique_id
df_clientes['customer_id'].nunique() - df_clientes['customer_unique_id'].nunique()

3345

In [16]:
# investigando customer_unique_id que aparecem mais de uma vez
contagem_clientes = df_clientes['customer_unique_id'].value_counts()
filtro_unique_id = contagem_clientes > 1

filtro_unique_id.value_counts()

count
False    93099
True      2997
Name: count, dtype: int64

In [17]:
contagem_clientes[filtro_unique_id]

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
1b6c7548a2a1f9037c1fd3ddfed95f33     7
                                    ..
2dcbfb99d95c7ba6eb992a98627f45b8     2
da29b8a3970adddb9ed2dc8a492065af     2
4c93744516667ad3b8f1fb645a3116a4     2
1175e95fb47ddff9de6b2b06188f7e0d     2
295c05e81917928d76245e842748184d     2
Name: count, Length: 2997, dtype: int64

In [18]:
# conferindo se os extras dos 2.997 clientes repetidos somam a diferença de 3.345 de customer_id para customer_unique_id
(contagem_clientes[filtro_unique_id] - 1).sum()

np.int64(3345)

In [19]:
# taxa de recompra
recompra = (filtro_unique_id.sum() / df_clientes['customer_unique_id'].nunique()*100).round(2)
print(f"A taxa de recompra é de: {recompra}%")

A taxa de recompra é de: 3.12%


### Achados na Tabela de Clientes:
- *customer_id* é único (99.441 linhas = 99.441 únicos) -> chave primária, 1 linha por pedido; ligação 1:1 com a tabela de pedidos via *customer_id*. Ao investigar na fonte (Kaggle) temos as seguintes definições:
    - *customer_id* -> "chave para a tabela de pedidos, cada pedido possui um *customer_id* exclusivo"
    - *customer_unique_id* -> "identificador único de um cliente"
    - Com isso, pode-se assumir que o identificador único do cliente está em *customer_unique_id*
- *customer_unique_id* possui 96.096 valores únicos, existe uma diferença de 3.345 para *customer_id*:
    - 2.997 clientes compraram mais de uma vez (até 17 vezes), gerando 3.345 *customer_id* adicionais. A relação foi confirmada por cálculo: a soma de (compras − 1) desses clientes é igual à diferença de 3.345
    - A taxa de recompra (calculada sobre todos os pedidos) está em 3,12%, demonstrando que a maioria dos clientes compra apenas uma vez. Na etapa de análise, será recalculada considerando apenas os pedidos 'delivered'
- *customer_zip_code_prefix* não possui valores nulos -> coluna fora das análises, sem impacto
- *customer_city* não possui valores nulos -> coluna fora das análises, sem impacto
- *customer_state* não possui valores nulos -> 27 valores únicos, condizente com Brasil (26 estados e 1 Distrito Federal)
    - Feita a conferência, as 27 siglas estão corretas
    - Estados mais populosos como SP e RJ possuem mais clientes do que estados menos populosos como AC, AP e RR
- Nas análises serão utilizados *customer_unique_id* para contar clientes e recompra; *customer_state* para análise regional

## 6.3 Tabela de Geolocalização

In [20]:
# recon(df_localizacao, "localizacao")

### Achados na Tabela de Geolocalização:
- Fora do escopo das perguntas do projeto. Não será utilizada nas análises.

## 6.4 Tabela de Itens dos Pedidos

In [21]:
recon(df_itens, "itens")

========== ITENS ==========
Shape (linhas e colunas): (112650, 7)
Linhas duplicadas: 0

Resumo por coluna:


,tipo,nulos,% nulos,únicos
order_id,object,0,0.0,98666
order_item_id,int64,0,0.0,21
product_id,object,0,0.0,32951
seller_id,object,0,0.0,3095
shipping_limit_date,object,0,0.0,93318
price,float64,0,0.0,5968
freight_value,float64,0,0.0,6999


Primeiras 5 linhas:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


Estatística descritiva:


,order_item_id,price,freight_value
count,112650.000000,112650.000000,112650.000000
mean,1.197834,120.653739,19.990320
std,0.705124,183.633928,15.806405
min,1.000000,0.850000,0.000000
25%,1.000000,39.900000,13.080000
50%,1.000000,74.990000,16.260000
75%,1.000000,134.900000,21.150000
max,21.000000,6735.000000,409.680000


### Investigação da Tabela de Itens dos Pedidos

In [22]:
# valores únicos de itens nos pedidos
df_itens['order_item_id'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21])

In [23]:
# checando pedidos com mais de 1 item
filtro_mais_itens = df_itens['order_item_id'] > 1
filtro_mais_itens.value_counts()

order_item_id
False    98666
True     13984
Name: count, dtype: int64

In [24]:
# checando duplicidade de pedidos (a combinação de order_id com order_item_id não pode ter duplicados) 
df_itens.duplicated(subset=['order_id', 'order_item_id']).sum()

np.int64(0)

In [25]:
# checando pedidos com produtos iguais
df_itens.duplicated(subset=['order_id', 'product_id']).sum()

np.int64(10225)

In [26]:
# testando com o maior pedido (21 itens)
filtro_pedido21 = df_itens['order_id'] == '8272b63d03f5f79c56e9e4120aec44ef'
df_itens[filtro_pedido21]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
57297,8272b63d03f5f79c56e9e4120aec44ef,1,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57298,8272b63d03f5f79c56e9e4120aec44ef,2,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57299,8272b63d03f5f79c56e9e4120aec44ef,3,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57300,8272b63d03f5f79c56e9e4120aec44ef,4,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57301,8272b63d03f5f79c56e9e4120aec44ef,5,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57302,8272b63d03f5f79c56e9e4120aec44ef,6,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57303,8272b63d03f5f79c56e9e4120aec44ef,7,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57304,8272b63d03f5f79c56e9e4120aec44ef,8,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57305,8272b63d03f5f79c56e9e4120aec44ef,9,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57306,8272b63d03f5f79c56e9e4120aec44ef,10,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89


In [27]:
# diferença de pedidos entre a Tabela de Pedidos e a Tabela de Itens dos Pedidos
df_pedidos['order_id'].nunique() - df_itens['order_id'].nunique()

775

In [28]:
mascara_sem_itens = ~df_pedidos['order_id'].isin(df_itens['order_id'])
mascara_sem_itens.value_counts()

order_id
False    98666
True       775
Name: count, dtype: int64

In [29]:
df_pedidos.loc[mascara_sem_itens, 'order_status'].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [30]:
# investigando price
df_itens['price'].describe()

count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64

In [31]:
df_itens['price'].nsmallest(10)

27652    0.85
48625    0.85
87081    0.85
57297    1.20
57298    1.20
57299    1.20
57300    1.20
57301    1.20
57302    1.20
57303    1.20
Name: price, dtype: float64

In [32]:
df_itens['price'].nlargest(10)

3556      6735.00
112233    6729.00
107841    6499.00
74336     4799.00
11249     4690.00
62086     4590.00
29193     4399.87
45843     4099.99
78310     4059.00
59137     3999.90
Name: price, dtype: float64

### Achados na Tabela de Itens dos Pedidos:
- *order_id* possui 98.666 valores únicos e na fonte é descrito como "identificador único do pedido", a tabela tem 112.650 linhas -> a diferença se dá para pedidos com mais de um item, com isso, o *order_id* se repete e o *order_item_id* vai sendo incrementado. A combinação de ambos não se repete, mostrando que a chave é composta por essas duas colunas
- *order_item_id* número de sequência do item no pedido (1 a 21); não é quantidade
    - Várias unidades do mesmo produto aparecem em linhas repetidas (10.225)
    - Testado com o pedido que contém 21 itens comprovando que várias unidades no mesmo pedido geram linhas repetidas (esse pedido tem 3 produtos diferentes)
- Existem 775 pedidos sem itens, porém nenhum deles possui o status 'delivered', portanto não terão impacto nas análises 
- *product_id* identifica o produto -> será usada para conectar na tabela de produtos
- *seller_id* identifica o vendedor -> essa coluna não será utilizada nas análises
- *shipping_limit_date* está como object e deveria ser data -> essa coluna não será utilizada, as análises serão com as datas da tabela de pedidos
- *price* contém os preços dos itens: 
    - 10 itens com menor preço entre 0,85 e 1,20 -> 10 itens com maior preço entre 3.999,90 e 6.735,00 -> por se tratar de um marketplace, não parece haver valores errados, portanto valores mantidos -> se houver necessidade, nas análises é possível juntar com a tabela de produtos e analisar se o preço é condizente com o produto
    - média de preços 120,65 vs. mediana 74,99 -> existe uma distância relevante entre média e mediana
    - a distribuição possui valores significativamente altos que puxam a média para cima com cauda longa à direita (assimetria positiva) -> verificar nos gráficos
    - considerando a assimetria, o preço típico no relatório deve ser com base na mediana 
    - o valor vendido por pedido a ser utilizado na análise é referente a soma de *price* dos itens de cada pedido
- *freight_value* preço do frete por item (se um pedido contiver mais de um item, o valor do frete é rateado entre eles) -> essa coluna não será utilizada nas análises -> para analisar valor de venda será usada a coluna *price*, pois o interesse da análise é no valor de produtos vendidos e não em quanto foi gasto com transporte


## 6.5 Tabela de Pagamentos

In [33]:
# recon(df_pagamentos, "pagamentos")

### Achados na Tabela de Pagamentos:
- Fora do escopo das perguntas do projeto. Não será utilizada nas análises.

## 6.6 Tabela de Avaliações

In [34]:
recon(df_avaliacoes, "avaliacoes")

========== AVALIACOES ==========
Shape (linhas e colunas): (99224, 7)
Linhas duplicadas: 0

Resumo por coluna:


,tipo,nulos,% nulos,únicos
review_id,object,0,0.00,98410
order_id,object,0,0.00,98673
review_score,int64,0,0.00,5
review_comment_title,object,87656,88.34,4527
review_comment_message,object,58247,58.70,36159
review_creation_date,object,0,0.00,636
review_answer_timestamp,object,0,0.00,98248


Primeiras 5 linhas:


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


Estatística descritiva:


,review_score
count,99224.000000
mean,4.086421
std,1.347579
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


### Investigação da Tabela de Avaliações

In [35]:
# quantidade dos reviews de acordo com nota e percentual sobre o total
proporcao_reviews = pd.DataFrame({
    "quantidade": df_avaliacoes["review_score"].value_counts(),
    "percentual": (df_avaliacoes["review_score"].value_counts(normalize=True)*100).round(2)})

proporcao_reviews.sort_index(ascending=False)

,quantidade,percentual
review_score,,
5,57328,57.78
4,19142,19.29
3,8179,8.24
2,3151,3.18
1,11424,11.51


In [36]:
# média das avaliações (não será utilizada - olhar célula com os achados)
df_avaliacoes['review_score'].mean().round(2)

np.float64(4.09)

In [37]:
# diferença na quantidade de registros (linhas) menos quantidade de review_id
linhas = df_avaliacoes.shape[0]
review_ids = df_avaliacoes['review_id'].nunique()

linhas - review_ids

814

In [38]:
review_id_repetido = df_avaliacoes['review_id'].value_counts() > 1
review_id_repetido.value_counts()

count
False    97621
True       789
Name: count, dtype: int64

In [39]:
order_id_repetido = df_avaliacoes['order_id'].value_counts() > 1
order_id_repetido.value_counts()

count
False    98126
True       547
Name: count, dtype: int64

In [40]:
# pedidos sem avaliação
mascara_sem_avaliacoes = ~df_pedidos['order_id'].isin(df_avaliacoes['order_id'])
mascara_sem_avaliacoes.value_counts()

order_id
False    98673
True       768
Name: count, dtype: int64

In [41]:
df_pedidos.loc[mascara_sem_avaliacoes, 'order_status'].value_counts()

order_status
delivered      646
shipped         75
canceled        20
unavailable     14
processing       6
invoiced         5
created          2
Name: count, dtype: int64

### Achados na Tabela de Avaliações:
- *review_creation_date* e *review_answer_timestamp* estão como object e deveriam ser data -> essas colunas não serão usadas na análise
- *review_comment_title* e *review_comment_message* possuem diversos valores nulos, porém esses textos não serão usados na análise -> é comum as pessoas avaliarem os pedidos e não deixar nenhum tipo de comentário
- *review_score* contém as notas de 1 a 5 e não possui valores nulos
    - 57.328 (57,78%) com 5 pontos; 19.142 (19,29%) com 4 pontos; 8.179 (8,24%) com 3 pontos; 3.151 (3,18%) com 2 pontos e 11.424 (11,51%) com 1 ponto
    - A distribuição não é gradual, é polarizada -> percentualmente, clientes altamente insatisfeitos estão em terceiro lugar no ranking com 11,51% das notas 1 -> hipótese: creio ser um comportamento normal nesse tipo de avaliação, gostei (5) ou não gostei (1) é mais fácil e rápido de avaliar do que colocar em uma escala de satisfação
    - Considerando que notas 4 e 5 representam clientes satisfeitos, temos 77,07% das avaliações nessa categoria 
    - Considerando que notas 1 e 2 representam clientes insatisfeitos temos 14,69% das avaliações nessa categoria
    - Considerando que nota 3 representa clientes neutros, temos 8,24% das avaliações nessa categoria
    - *review_score* é uma variável qualitativa ordinal, há ordem, mas não sabemos “quanto” uma categoria é maior que a outra, por isso podemos ordenar, mas não é aconselhado usar a média calculada anteriormente (4,09) -> posto isso, vou trabalhar com as categorias: satisfeito (notas 4 e 5), neutro (nota 3) e insatisfeito (notas 1 e 2)
- *order_id* possui o identificador único da ordem -> 98.673 valores únicos
- *review_id* possui o identificador único da avaliação -> 98.410 valores únicos
- Relação entre pedidos e avaliações não é 1:1:
    - 789 avaliações estão ligadas a mais de um pedido -> não causa duplicidade na análise por pedido, portanto serão mantidas
    - 547 pedidos possuem mais de uma avaliação (0,55 % dos pedidos avaliados) -> geram linhas duplicadas na junção com a tabela de pedidos -> serão excluídos das análises de satisfação
- Dos 768 pedidos que não possuem avaliação, 646 estão com status 'delivered' (0,7% dos pedidos entregues) -> esses pedidos serão excluídos nas análises de satisfação, mas serão mantidos nas análises de venda
- Importante: todos esses cálculos percentuais foram feitos utilizando os dados 'raw' e serão recalculados após a limpeza usando apenas pedidos 'delivered'

## 6.7 Tabela de Produtos

In [42]:
recon(df_produtos, "produtos")

========== PRODUTOS ==========
Shape (linhas e colunas): (32951, 9)
Linhas duplicadas: 0

Resumo por coluna:


,tipo,nulos,% nulos,únicos
product_id,object,0,0.00,32951
product_category_name,object,610,1.85,73
product_name_lenght,float64,610,1.85,66
product_description_lenght,float64,610,1.85,2960
product_photos_qty,float64,610,1.85,19
product_weight_g,float64,2,0.01,2204
product_length_cm,float64,2,0.01,99
product_height_cm,float64,2,0.01,102
product_width_cm,float64,2,0.01,95


Primeiras 5 linhas:


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


Estatística descritiva:


,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000


### Investigação da Tabela de Produtos

In [43]:
# quantidade e percentual de produtos por categorias não nulos

categorias = pd.DataFrame({
    "quantidade": df_produtos['product_category_name'].value_counts(),
    "percentual": (df_produtos['product_category_name'].value_counts(normalize=True)*100).round(2)})

categorias

,quantidade,percentual
product_category_name,,
cama_mesa_banho,3029,9.37
esporte_lazer,2867,8.86
moveis_decoracao,2657,8.22
beleza_saude,2444,7.56
utilidades_domesticas,2335,7.22
...,...,...
fashion_roupa_infanto_juvenil,5,0.02
casa_conforto_2,5,0.02
pc_gamer,3,0.01


In [44]:
categorias.head(10)

,quantidade,percentual
product_category_name,,
cama_mesa_banho,3029,9.37
esporte_lazer,2867,8.86
moveis_decoracao,2657,8.22
beleza_saude,2444,7.56
utilidades_domesticas,2335,7.22
automotivo,1900,5.87
informatica_acessorios,1639,5.07
brinquedos,1411,4.36
relogios_presentes,1329,4.11


In [45]:
# percentual das top 5 categorias
categorias['percentual'].head().sum()

np.float64(41.23)

In [46]:
# percentual das top 10 categorias
categorias['percentual'].head(10).sum()

np.float64(64.15)

In [47]:
# análise dos produtos sem categoria
mascara_sem_categoria = df_produtos['product_category_name'].isna()
produtos_sem_categoria = df_produtos.loc[mascara_sem_categoria, 'product_id']

In [48]:
mascara_itens_sem_categoria = df_itens['product_id'].isin(produtos_sem_categoria)

In [49]:
mascara_sem_categoria.sum()

np.int64(610)

In [50]:
mascara_itens_sem_categoria.sum()

np.int64(1603)

In [51]:
(mascara_itens_sem_categoria.mean()*100).round(2)

np.float64(1.42)

In [52]:
# percentual do valor vendido de itens sem categoria
(df_itens.loc[mascara_itens_sem_categoria, 'price'].sum() / df_itens['price'].sum()*100).round(2)

np.float64(1.32)

### Achados na Tabela de Produtos:
- *product_id* representa o identificador único do produto -> (32.951 linhas = 32.951 únicos) -> chave primária, 1 linha por produto; ligação 1:N com a tabela de itens via *product_id*
- *product_category_name* é a categoria 'raiz' do produto (em português) -> existem 73 categorias únicas
    - Top 10 categorias do catálogo (excluindo nulos): cama_mesa_banho (9,37%), esporte_lazer (8,86%), moveis_decoracao (8,22%), beleza_saude (7,56%), utilidades_domesticas (7,22%), automotivo (5,87%), informatica_acessorios (5,07%), brinquedos (4,36%), relogios_presentes (4,11%) e telefonia (3,51%)
    - Top 5 categorias do catálogo representam 41,23%; Top 10 categorias do catálogo representam 64,15%
    - Existem 610 produtos sem categoria (1,85% do catálogo) -> eles correspondem a 1,42% dos itens vendidos e 1,32% do valor vendido -> serão excluídos na análise por categoria e mantidos no total de vendas
- *product_name_lenght*, *product_description_lenght*, *product_photos_qty*, *product_weight_g*, *product_length_cm*, *product_height_cm* e *product_width_cm* -> colunas com descrições e medidas que não serão usadas nas análises

## 6.8 Tabela de Vendedores

In [53]:
# recon(df_vendedores, "vendedores")

### Achados na Tabela de Vendedores:
- Fora do escopo das perguntas do projeto. Não será utilizada nas análises.

# 7.0 Resumo das Decisões

## 7.1 Conversões de tipo

- Colunas 'object' que serão convertidas em 'date':
    - Tabela de Pedidos: *order_purchase_timestamp*, *order_estimated_delivery_date* e *order_delivered_customer_date*

## 7.2 Filtros Gerais

- Todas as análises usarão pedidos com status 'delivered' na Tabela de Pedidos (mais de 97% dos casos)
- Período de análise: 2017-01 a 2018-08 (meses com volume consistente); crescimento medido pelo acumulado jan–ago/2017 vs. jan–ago/2018

## 7.3 Exclusões por Análise

- Prazo de entrega: 8 pedidos 'delivered' sem *order_delivered_customer_date*
- Satisfação:
    - Pedidos 'delivered' sem avaliação (646 na base bruta; 643 após o filtro de período): ficam fora da satisfação naturalmente, mantidos nas vendas
    - Pedidos com mais de uma avaliação (547 na base bruta; 523 após os filtros): decisão revista na limpeza -> removidos diretamente da tabela de avaliações, pois ela só é usada nas análises de satisfação e a remoção não afeta as vendas
- Categorias: itens de 610 produtos sem categoria (1,42% dos itens; 1,32% do valor vendido), mantidos no total de vendas
- Vendas: nenhuma exclusão além do filtro 'delivered' (os 775 pedidos sem itens não são 'delivered')

## 7.4 Definições de Métricas

- Data da venda: *order_purchase_timestamp*, agregada por mês
- Pedidos: contagem de *order_id* distintos
- Valor vendido: soma de *price* dos itens de cada pedido, sem frete (valor dos produtos vendidos, não é receita da Olist)
- Preço típico: mediana de *price* (distribuição com assimetria positiva)
- Ticket por pedido: mediana, a confirmar na análise
- Clientes: contagem de *customer_unique_id* distintos
- Taxa de recompra: % de *customer_unique_id* com mais de um pedido 'delivered'
- Satisfação: % de avaliações por grupo: satisfeito (4 e 5), neutro (3), insatisfeito (1 e 2); média não utilizada (variável ordinal)
- Tempo de entrega: dias entre *order_purchase_timestamp* e *order_delivered_customer_date*
- Atraso: dias entre *order_estimated_delivery_date* e *order_delivered_customer_date* (positivo = entregue após o prometido)
- Região: *customer_state* (UF do cliente)
- Categoria: *product_category_name*

## 7.5 Tabelas e Colunas Fora do Escopo

- Tabelas:
    - Geolocalização, Pagamentos e Vendedores
- Colunas:
    - Tabela de Pedidos: *order_approved_at* e *order_delivered_carrier_date*
    - Tabela de Clientes: *customer_zip_code_prefix* e *customer_city*
    - Tabela de Itens dos Pedidos: *seller_id*, *shipping_limit_date* e *freight_value*
    - Tabela de Avaliações: *review_creation_date*, *review_answer_timestamp*, *review_comment_title* e *review_comment_message*
    - Tabela de Produtos: *product_name_lenght*, *product_description_lenght*, *product_photos_qty*, *product_weight_g*, *product_length_cm*, *product_height_cm* e *product_width_cm*